# PubMed Evidence Assistant - Notebook 3: serve it (API + public demo) on a free Colab T4

This runs the real service from the repo: **FastAPI** (`/ask`, `/feedback`, `/stats`, `/health`) in a background thread, the local Qwen model on the T4, and the **Gradio** demo with a public share link (free, no account).

**Before you start**
1. Notebooks 01 and 02 must have been run, so `MyDrive/pubmed_rag/results/` contains the `score_train_rag_whole_*.jsonl` files (they are the calibrator's training data).
2. Put the repo folder in Drive at `MyDrive/pubmed_rag/pubmed-evidence-assistant/` (upload the folder, or `git clone` it there).
3. Runtime -> T4 GPU.

The public link only lives while this notebook runs. That is fine for recording a demo video or showing it in an interview: open the notebook, run all cells, share the link.

In [1]:
!pip -q install fastapi uvicorn rank_bm25 gradio requests datasets sentence-transformers accelerate scikit-learn

In [8]:
import glob, os, shutil

dest = "/content/drive/MyDrive/pubmed_rag/pubmed-evidence-assistant"
found = glob.glob("/content/*assisstant")
print("found:", found)
assert found, "clone folder not found in /content - run the git clone cell again first"
if os.path.exists(dest):
    shutil.rmtree(dest)
shutil.copytree(found[0], dest)
print(sorted(os.listdir(dest)))

found: ['/content/irsaaziz-spec-irsaaziz-spec-pubmed_evidence-assisstant']
['.git', '.github', 'Dockerfile', 'LICENSE', 'README.md', 'app', 'notebooks', 'pyproject.toml', 'requirements-dev.txt', 'requirements-ml.txt', 'requirements-ui.txt', 'requirements.txt', 'scripts', 'src', 'tests']


In [9]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/pubmed_rag
!git clone https://github.com/irsaaziz-spec/irsaaziz-spec-irsaaziz-spec-pubmed_evidence-assisstant
!ls /content/drive/MyDrive/pubmed_rag/pubmed-evidence-assistant

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'irsaaziz-spec-irsaaziz-spec-pubmed_evidence-assisstant' already exists and is not an empty directory.
app	    notebooks	    requirements-dev.txt  requirements-ui.txt  tests
Dockerfile  pyproject.toml  requirements-ml.txt   scripts
LICENSE     README.md	    requirements.txt	  src


In [10]:
import json, os, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/pubmed_rag")
RESULTS = BASE / "results"

# find the project folder wherever it is (the one that contains src/pubmed_rag)
found = sorted((p.parent for p in BASE.rglob("pyproject.toml") if (p.parent / "src" / "pubmed_rag").exists()),
               key=lambda p: len(p.parts))
assert found, f"No project found under {BASE}. Run the git clone cell first (see below)."
REPO = found[0]
print("project folder:", REPO)
if len(found) > 1:
    print("note: more than one copy found, using the first:", [str(f) for f in found])

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "app"))
os.chdir(REPO)
INDEX = REPO / "index"
print("results folder exists:", RESULTS.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project folder: /content/drive/MyDrive/pubmed_rag/pubmed-evidence-assistant
results folder exists: True


## 1. Build the index (once)

Downloads PubMedQA, embeds ~5,000 abstracts with bge-base on the GPU and writes `index/` into the repo folder on Drive. This is the **same corpus as the evaluation** (1,000 labelled abstracts + 4,000 distractors). Skipped if it already exists.

In [11]:
from pubmed_rag.build import build_index

if not (INDEX / "embeddings.npy").exists():
    build_index(INDEX, n_distractors=4000)
meta = json.loads((INDEX / "meta.json").read_text())
print({k: meta[k] for k in ("n_docs", "n_questions", "embed_model", "built_at")})

README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

{'n_docs': 5000, 'n_questions': 1000, 'embed_model': 'BAAI/bge-base-en-v1.5', 'built_at': '2026-11-22T19:52:08'}


## 2. Fit the calibrated decision head (once)

Uses the 300 training-question scores from Notebook 02 (`rag_whole` setup: whole abstracts, dense, k=5, i.e. exactly what the service does). Both variants are saved; the service uses `unweighted` (best accuracy on dev) unless you set `PMRAG_CALIBRATOR_VARIANT=balanced`.

In [12]:
from pubmed_rag.calibrate import fit_from_jsonl

if not (INDEX / "calibrator.json").exists():
    files = sorted(RESULTS.glob("score_train_rag_whole_*.jsonl"), key=lambda p: p.stat().st_mtime)
    assert files, f"No score_train_rag_whole_*.jsonl in {RESULTS}. Run Notebook 02 (Part B) first."
    payload = fit_from_jsonl(files[-1:], INDEX / "calibrator.json")
    print("fitted on", payload["n_train"], "rows from", files[-1].name)
else:
    print("calibrator.json already exists")

fitted on 300 rows from score_train_rag_whole_e920e2.jsonl


## 3. Load the service (model download: a few minutes the first time)

In [13]:
from pubmed_rag.api import build_assistant, create_app
from pubmed_rag.config import Settings

settings = Settings(index_dir=INDEX, backend="local", retrieval_mode="dense", k=5, log_dir=REPO / "logs")
assistant = build_assistant(settings)
print("backend:", assistant.backend.name, "| retrieval:", assistant.mode, "| calibrated:", assistant.calibrator is not None,
      "| corpus:", len(assistant.retriever.corpus))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

backend: local | retrieval: dense | calibrated: True | corpus: 5000


### Sanity check: does the service reproduce the notebook's retrieval numbers?

Notebook 01 measured dense whole-abstract retrieval at hit@5 about 0.99 on all 1,000 questions. If the service's index or embedder were wrong, this number would collapse. It is a regression check for the deployment, not a new result.

In [14]:
from pubmed_rag.evaluate import retrieval_metrics

questions = [json.loads(l) for l in (INDEX / "questions.jsonl").read_text().splitlines() if l.strip()]
m = retrieval_metrics(assistant.retriever, questions, "dense", k=5)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in m.items()})
assert m["hit_at_k"] > 0.97, "retrieval looks broken: check that index/ was built with the same embed model"

{'mode': 'dense', 'k': 5, 'n': 1000, 'hit_at_k': 0.991, 'mrr': 0.983, 'ms_per_query': 19.5}


## 4. Start the API in the background and smoke-test it

In [15]:
import requests
from pubmed_rag.serve import start_background

server, thread = start_background(create_app(settings, assistant), port=8000)
API = "http://127.0.0.1:8000"
print(requests.get(f"{API}/health").json())

r = requests.post(f"{API}/ask", json={"question": "Does high blood pressure reduce the risk of chronic low back pain?"}).json()
print(r["decision"], {k: round(v, 3) for k, v in r["probabilities"].items()}, "| top abstract:", r["evidence"][0]["doc_id"])
print("timings (ms):", r["timings_ms"])

{'status': 'ok', 'version': '0.1.0', 'backend': 'local', 'retrieval_mode': 'dense', 'calibrated': True, 'corpus_size': 5000}
yes {'maybe': 0.213, 'no': 0.33, 'yes': 0.457} | top abstract: 24019262
timings (ms): {'retrieval': 32.9, 'scoring': 1240.6, 'explain': 0.0, 'total': 1273.6}


## 5. Launch the public demo

The cell below stays running (that is what keeps the link alive). Copy the `https://....gradio.live` URL it prints. To stop: interrupt the cell.

In [16]:
from demo import build_demo

demo = build_demo(API)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b22473159449b8a9c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b22473159449b8a9c9.gradio.live


## 6. After a demo session: what the service recorded (AIOps)

Run these after interrupting the cell above. The API has been running in the background, so `/stats` and the logs reflect everything you did in the demo. Questions are logged only as hashes unless you set `log_questions=True`.

In [17]:
print(requests.get(f"{API}/stats").json())
print(open(REPO / "logs" / "requests.jsonl").read().splitlines()[-1][:300])
fb = REPO / "logs" / "feedback.jsonl"
print("feedback rows:", len(fb.read_text().splitlines()) if fb.exists() else 0)

{'requests_total': 8, 'window': 8, 'p50_ms': 1143.8, 'p95_ms': 9503.4}
{"ts": 1795378108.4954937, "request_id": "ecbfad4a44b8", "question_hash": "e54544e1f63b", "decision": "no", "backend": "local", "mode": "dense", "k": 6, "timings_ms": {"retrieval": 16.2, "scoring": 1328.5, "explain": 8045.1, "total": 9389.8}, "warning": null, "top_doc": "24019262"}
feedback rows: 3


## Record the demo

Screen-record about 60-90 seconds: ask one of the example questions, show the verdict, the probabilities and the cited PubMed abstracts, click a PubMed link, use the feedback button, then show `/stats`. Add the video (or a GIF) to the repo README.